# FMN Project 1 — Inventory Risk and Attention Engine

This notebook turns demand forecasts and replenishment behaviour into a planner facing inventory risk signal. The engine separates analytical risk from planner attention and uses expected delivery timing to reduce avoidable alerts.

**As of:** 29 June 2026

The risk engine is deterministic. It does not place orders or recommend an automated order quantity.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('..')
data_path = ROOT / 'data' / 'processed' / 'clean_daily_data.csv'
df = pd.read_csv(data_path, parse_dates=['date'])
df = df.sort_values(['sku_id', 'date']).reset_index(drop=True)
df.shape

### Finding
The prepared dataset contains 4,536 rows covering 28 SKUs through 29 June 2026. The risk engine therefore operates on the same cleaned data foundation used for forecasting.

## 1. Risk design

Core calculation:

`projected stock = current stock + expected receipts − forecast demand`

Risk states:
- **Critical:** current stock is zero, or stock is projected to reach zero before expected replenishment.
- **Watch:** projected stock falls below the safety buffer within the review window.
- **Healthy:** projected stock covers the relevant demand with replenishment protection.
- **Overstock:** projected stock materially exceeds one replenishment cycle plus buffer.

Initial product parameters tested: review windows of 1, 3 and 7 days and uncertainty multipliers of 0, 0.25, 0.5 and 0.75.

In [ ]:
comparison = pd.read_csv(ROOT / 'artifacts' / 'evaluation' / 'risk_model_comparison.csv')
sensitivity = pd.read_csv(ROOT / 'artifacts' / 'evaluation' / 'risk_parameter_sensitivity.csv')
comparison[comparison['lead_time'].astype(str) == 'All']

### Finding
The baseline rule has very high recall but creates a large attention load. The candidate risk engine reduces attention volume and false alert rate while improving precision, but it does not fully preserve the approximately 0.90 recall objective across the development backtest. This tradeoff must remain visible rather than being hidden.

In [ ]:
sensitivity[['review_window_days','uncertainty_multiplier','recall','precision','false_alert_rate','attention_volume']]


### Finding
A 3 day review window with a 0.5 uncertainty multiplier gives a useful MVP tradeoff: about 86% recall, 37% precision, 18% false alert rate and 26% attention volume. The 7 day window can raise recall, but does so with materially higher alert burden.

**MVP decision:** retain the PRD default of a 3 day review window and 0.5 uncertainty multiplier, while showing the validation metrics transparently. This is a development choice, not a universal inventory constant.

## 2. Current attention assessment

The runtime assessment is generated as of 29 June 2026 using the pooled LightGBM forecast and historical replenishment rhythm. Expected delivery is inferred from historical receipt cadence, not future actual receipts.

In [ ]:
current = pd.read_csv(ROOT / 'artifacts' / 'evaluation' / 'risk_current_assessment.csv', parse_dates=['date','expected_delivery_date'])
attention = current[current['risk_state'].isin(['Critical','Watch'])].copy()
attention[['sku_id','risk_state','current_stock','lead_time_days','forecast_lead_time_demand_model','coverage_days','expected_delivery_date','days_to_projected_stockout','projected_unmet_units','drivers','recommendation']].sort_values(['risk_state','projected_unmet_units'], ascending=[True,False])

### Finding
The current assessment identifies a focused set of urgent SKUs rather than flagging every low coverage SKU. In particular, SKU-1010 has less than one day of current cover but an expected delivery the next day, so it is not treated as Critical solely from a static stock versus demand comparison. This directly addresses the false alarm problem found during discovery.

## 3. Business interpretation

The Attention Center should rank Critical items by projected unmet units, then use days to projected stockout, lead time, delivery timing and uncertainty as supporting context.

Recommended actions are deliberately investigation oriented:
- Review replenishment and consider expediting.
- Investigate the expected delivery.
- Check whether an order is already in transit.
- Review demand assumptions.
- Check data quality before acting.
- Investigate excess inventory before the next replenishment.

The system does not calculate an autonomous purchase quantity.

## 4. Limitations and next decision gate

1. The assessment dataset does not contain open purchase orders, so expected delivery is inferred from historical receipt rhythm. Production deployment should replace this with actual open order data.
2. Risk parameter tuning and backtesting are based on this assessment dataset. The final application should keep the validation page visible.
3. The current recall tradeoff is not perfect. If additional time is available, risk calibration should be revisited using a stronger event definition and a separate holdout period.
4. New SKUs require wider uncertainty and explicit limited history/data issue badges.

**Next stage:** create the shared `SkuAssessment` contract, attention ranking output and then connect the deterministic engine to the Streamlit product and grounded explanation layer.